# Challenge 2: Optimization of Battery Usage in the Installation

In this notebook, we address Objective 2 from the Repsol IE Sustainability Challenge:

- **Objective:** Optimize the use of a theoretical battery (100 kWh capacity, 100 kW charge/discharge, one cycle per day) to maximize self‑consumption of solar energy and reduce grid dependence.

We will:

1. Load the necessary datasets: solar generation potential (from Challenge 1), actual photovoltaic consumption, and grid consumption.
2. Compute the surplus solar energy available each hour.
3. Simulate battery operation: determine when to charge (using surplus solar) and when to discharge (to substitute grid energy, especially when carbon intensity is high).
4. Compute the Self‑Consumption Ratio (Ra) and (optionally) an estimate of CO₂ avoided.
5. Export the battery simulation predictions for further evaluation.

Let's begin!

In [1]:
import pandas as pd
import numpy as np

## 1) Data Loading & Preparation, surplus calculation

We load the following datasets:

- **Solar Generation Potential:** (predicted in Challenge 1) from your previous work (e.g. stored in `cleanDatav3.csv` or an output file from Challenge 1).
- **Actual Photovoltaic Consumption:** (e.g., from `Consumido_Fotovoltaica.csv`).
- **Grid Consumption:** (e.g., from `Consumo.csv`).

We also convert the timestamps to local time (e.g., Europe/Madrid) and filter the September period (which must have exactly 720 hourly rows).


We calculate the surplus solar energy as the difference between the predicted solar generation and the actual photovoltaic consumption. (Any negative surplus is set to zero since surplus only exists when generation exceeds consumption.)

In [50]:
df_pred = pd.read_csv("september_predictions3.csv")
df_pred=df_pred.rename(columns={'datetime':'Datetime'})
df_pred['Datetime'] = pd.to_datetime(df_pred['Datetime'])
df_pred = df_pred.rename(columns={"KWH_ENERGIA": "predicted_generation"})

# ---------------------------
# 2. Load Actual Photovoltaic Consumption
# ---------------------------
df_pv = pd.read_csv("Consumo_fotovoltaica.csv")
df_pv['FECHA']=df_pv['FECHA'].str.slice(0,-6)
df_pv['FECHA'] = pd.to_datetime(df_pv['FECHA'])
df_pv = df_pv.rename(columns={"FECHA": "Datetime", "TOTAL_KWH_ENERGIA": "actual_pv"})

# ---------------------------
# 3. Merge Predicted and Actual PV Data
# ---------------------------
df_solar = pd.merge(df_pred, df_pv, on="Datetime", how="inner")
df_solar['surplus'] = (df_solar['pv_generation_pred'] - df_solar['actual_pv']).clip(lower=0)

# ---------------------------
# 4. Load CO₂ Hourly Data
# ---------------------------
df_co2 = pd.read_csv("ES_2024_hourly.csv")
# Rename "Datetime (UTC)" to "Datetime" for merging
df_co2 = df_co2.rename(columns={"Datetime (UTC)": "Datetime", 
                                "Carbon Intensity gCO₂eq/kWh (direct)": "Carbon_Intensity"})
df_co2['Datetime'] = pd.to_datetime(df_co2['Datetime'])

# ---------------------------
# 5. Merge Solar Data with CO₂ Data
# ---------------------------
df_merged = pd.merge(df_solar, df_co2[['Datetime', 'Carbon_Intensity']], on="Datetime", how="inner")

# ---------------------------
# 6. Filter for September 2024
# ---------------------------
start_date = pd.Timestamp("2024-09-01")
end_date = pd.Timestamp("2024-09-30 23:59:59")
df_sep = df_merged[(df_merged['Datetime'] >= start_date) & (df_merged['Datetime'] <= end_date)].copy()



## 3) Battery Simulation

We simulate a theoretical battery with the following characteristics:

- **Capacity:** 100 kWh
- **Max Charge/Discharge Power:** 100 kW (i.e. it can charge or discharge up to 100 kWh in one hour)
- **One Charge/Discharge Cycle Per Day:** Once per day, the battery is allowed to charge and then discharge.

Our simulation follows these steps for each day in September:

1. **Charging:** When surplus energy is available, charge the battery without exceeding capacity.
2. **Discharging:** At the hour(s) with the highest grid carbon intensity (if grid consumption is high), discharge the battery to substitute grid energy. (For this notebook, we use a simplified rule: discharge once per day using the available battery energy.)
3. Track the energy charged, lost (if battery is full), and energy recovered via discharge.

For simplicity, we simulate a basic strategy where the battery charges during the hours of maximum surplus and then discharges at a predefined peak demand hour (this can be refined further).

In [51]:
# 7. Initialize Battery Simulation Columns
# ---------------------------
df_sep['charge'] = 0.0
df_sep['discharge'] = 0.0
df_sep['SoC'] = 0.0  # State-of-Charge (kWh)
df_sep['date'] = df_sep['Datetime'].dt.date  # for grouping by day

# ---------------------------
# 8. Battery Parameters
# ---------------------------
battery_capacity = 100.0   # kWh
max_charge_rate = 100.0    # kWh per hour
max_discharge_rate = 100.0 # kWh per hour

In [52]:
# 9. Battery Simulation Function (per day)
# ---------------------------
def simulate_day(day_df):
    day_df = day_df.sort_values(by='Datetime').copy()
    soc = 0.0  # initial state-of-charge for the day
    day_df['SoC'] = 0.0
    day_df['charge'] = 0.0
    day_df['discharge'] = 0.0
    
    # Charging Phase: charge battery using available surplus until full
    for idx, row in day_df.iterrows():
        if soc < battery_capacity:
            charge_amount = min(row['surplus'], battery_capacity - soc, max_charge_rate)
            soc += charge_amount
            day_df.at[idx, 'charge'] = charge_amount
        day_df.at[idx, 'SoC'] = soc
    
    # Compute a CO₂ "cost" for each hour as an indicator (if grid consumption were present)
    # Here, since we want to know when discharging gives the highest CO₂ savings,
    # we use battery discharge * Carbon_Intensity as the potential saving.
    # For selection, we consider the hour with the highest Carbon_Intensity,
    # because discharging then avoids a higher emissions factor.
    discharge_idx = day_df['Carbon_Intensity'].idxmax()
    discharge_time = day_df.loc[discharge_idx, 'Datetime']
    discharge_amount = min(soc, max_discharge_rate)
    day_df.at[discharge_idx, 'discharge'] = discharge_amount
    
    # Update SoC for all hours at or after the discharge time to reflect the battery discharge
    for idx, row in day_df.iterrows():
        if row['Datetime'] >= discharge_time:
            day_df.at[idx, 'SoC'] = max(day_df.at[idx, 'SoC'] - discharge_amount, 0)
    
    return day_df

In [53]:
# 10. Apply Simulation to Each Day in September 2024
# ---------------------------
simulated_days = []
for date, group in df_sep.groupby('date'):
    simulated_day = simulate_day(group)
    simulated_days.append(simulated_day)

df_simulated = pd.concat(simulated_days).sort_index().reset_index(drop=True)

## 4) Compute Business Metrics

### Self‑Consumption Ratio (Ra)

We define Ra as the ratio of the solar energy used (both directly and via battery) to the total solar generation potential.

For example:

```
Ra = (Direct Solar Consumption + Energy Recovered from Battery) / Total Solar Generation
```

You can then calculate this for the entire month of September.

### CO₂ Reduction (CO₂ev)

If you have grid consumption and carbon intensity data, you can compute the CO₂ emissions avoided by substituting grid energy with battery discharge. For simplicity, here we outline the calculation as:

```
CO2ev = Total CO2 (baseline grid consumption) - Total CO2 (after battery discharge)
```

For now, we focus on computing Ra.

In [55]:
# 12. Calculate Self-Consumption Ratio (Ra)
# ---------------------------
# Total predicted solar generation (baseline potential)
total_solar_generated = df_simulated['pv_generation_pred'].sum()
# Total solar used directly is the sum of actual PV consumption
total_solar_used = df_simulated['actual_pv'].sum()
# Total energy recovered via battery discharge
total_energy_recovered = df_simulated['discharge'].sum()

Ra = (total_solar_used + total_energy_recovered) / total_solar_generated
print(f"Self-Consumption Ratio (Ra): {Ra:.4f}")

Self-Consumption Ratio (Ra): 1.1107


In [61]:
# 12. Environmental Impact Analysis (CO₂ Savings)
# ---------------------------
df_simulated['CO2_Avoided'] = df_simulated['discharge'] * df_simulated['Carbon_Intensity']+df_simulated['pv_generation_pred']*df_simulated['Carbon_Intensity']

total_CO2_avoided = df_simulated['CO2_Avoided'].sum()
print(f"Total CO₂ avoided (gCO₂eq) for September 2024: {total_CO2_avoided:.4f}")

Total CO₂ avoided (gCO₂eq) for September 2024: 1624856.1807


## 5) Export Results

Finally, we export the September predictions (with battery simulation results) to a CSV file. Ensure that the exported file has exactly 720 rows with the correct local timestamps.

In [62]:
# Make sure the data is sorted by datetime
df_simulated.sort_values('Datetime', inplace=True)

# Verify row count (should be 720)
print('Number of rows in September simulation:', df_simulated.shape[0])

# Export final predictions
#export_cols = ['datetime', 'pv_generation_pred', 'TOTAL_KWH_ENERGIA', 'energy_discharged', 'solar_used']
df_simulated.to_csv('september_predictions_battery_simulation.csv', index=False)
print("Predictions exported to 'september_predictions_battery_simulation.csv'")

Number of rows in September simulation: 720
Predictions exported to 'september_predictions_battery_simulation.csv'


In [63]:
df_simulated.head(20)

,Datetime,pv_generation_pred,actual_pv,surplus,Carbon_Intensity,charge,discharge,SoC,date,CO2_Avoided
0,2024-09-01 00:00:00,0.116403,0.00,0.116403,170.04,0.116403,0.0,0.116403,2024-09-01,19.793127
1,2024-09-01 01:00:00,0.023055,0.00,0.023055,175.13,0.023055,0.0,0.139458,2024-09-01,4.037628
2,2024-09-01 02:00:00,0.010746,0.00,0.010746,175.53,0.010746,0.0,0.150204,2024-09-01,1.886220
3,2024-09-01 03:00:00,0.002073,0.00,0.002073,174.22,0.002073,0.0,0.152277,2024-09-01,0.361189
4,2024-09-01 04:00:00,0.003082,0.00,0.003082,175.05,0.003082,0.0,0.155358,2024-09-01,0.539427
5,2024-09-01 05:00:00,0.013595,0.00,0.013595,170.19,0.013595,0.0,0.168953,2024-09-01,2.313696
6,2024-09-01 06:00:00,0.111413,0.00,0.111413,159.95,0.111413,0.0,0.280366,2024-09-01,17.820476
7,2024-09-01 07:00:00,0.208219,0.00,0.208219,123.35,0.208219,0.0,0.488585,2024-09-01,25.683841
8,2024-09-01 08:00:00,0.630956,0.00,0.630956,81.80,0.630956,0.0,1.119541,2024-09-01,51.612211
9,2024-09-01 09:00:00,5.389729,6.05,0.000000,66.29,0.000000,0.0,1.119541,2024-09-01,357.285156


# Objective 3

## Wrap-Up & Conclusion

In this notebook we:

1. Loaded solar generation potential predictions, actual photovoltaic consumption, and grid consumption data.
2. Engineered time-based, meteorological, and lag features.
3. Calculated surplus solar energy and simulated a theoretical battery operation (charge/discharge once per day).
4. Computed the Self‑Consumption Ratio (Ra) as a key business metric.
5. Exported the final September prediction results to CSV for submission.

While this is a simplified battery simulation, further refinements can be made by:

- Enhancing the battery simulation strategy (e.g., optimizing charging/discharging times more dynamically).
- Incorporating grid consumption, carbon intensity, and a detailed CO₂ reduction calculation.
- Improving feature engineering (e.g., adding additional lags, rolling means, or external factors like holidays).

Keep iterating to achieve the target MAE and business metrics. Good luck!